In [1]:
!pip install -q opensmile tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 632.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 13.8 MB/s eta 0:00:00


In [2]:
import os
import glob
import opensmile
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score
import warnings

warnings.filterwarnings('ignore')

print("All imports done.")

All imports done.


In [16]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/AudialMind'

speech_files = sorted(glob.glob(
    os.path.join(DATA_DIR, '**', '*.wav'), recursive=True))
speech_files = [f for f in speech_files
                if os.path.basename(f).split('-')[0] == '03']

print(f"Speech files found: {len(speech_files)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Speech files found: 1440


In [17]:
EMOTION_MAP = {
    '01': 'neutral', '02': 'calm',     '03': 'happy',
    '04': 'sad',     '05': 'angry',    '06': 'fearful',
    '07': 'disgust', '08': 'surprised'
}

def get_emotion(path):
    parts = os.path.splitext(os.path.basename(path))[0].split('-')
    return EMOTION_MAP[parts[2]]

print("Label parser ready.")
print("Test:", get_emotion('03-01-05-01-01-01-24.wav'))  # angry

Label parser ready.
Test: angry


Step1

In [18]:
import opensmile
import os
import inspect

# Method 1 — find via package location
opensmile_path = os.path.dirname(inspect.getfile(opensmile))
print(f"openSMILE package path:\n{opensmile_path}\n")

# Search for config files
import glob
conf_files = glob.glob(os.path.join(opensmile_path, '**', '*.conf'), recursive=True)

print(f"Config files found: {len(conf_files)}")
for f in conf_files[:10]:   # print first 10
    print(f"  {f}")

openSMILE package path:
/usr/local/lib/python3.12/dist-packages/opensmile

Config files found: 13
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/compare/ComParE_2016.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/is09-13/IS13_ComParE.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/is09-13/IS10_paraling_compat.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/is09-13/IS13_ComParE_Voc.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/is09-13/IS11_speaker_state.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/is09-13/IS09_emotion.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/is09-13/IS12_speaker_trait_compat.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/emobase/emobase.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/gemaps/v01a/GeMAPSv01a.conf
  /usr/local/lib/python3.12/dist-packages/opensmile/core/config/

In [19]:
# Find egemaps config specifically
egemaps_conf = [f for f in conf_files
                if 'egemaps' in f.lower()
                or 'eGeMAPSv02' in f]

if egemaps_conf:
    print(f"Found: {egemaps_conf[0]}\n")
    with open(egemaps_conf[0], 'r') as f:
        print(f.read()[:3000])
else:
    # Try direct path
    direct_path = os.path.join(opensmile_path,
                  'config', 'egemaps', 'v02',
                  'eGeMAPSv02.conf')
    if os.path.exists(direct_path):
        print(f"Found at: {direct_path}")
        with open(direct_path, 'r') as f:
            print(f.read()[:3000])
    else:
        print("Not found — skipping to Cell 7")
        print("This is just for reference, not required to run")

Found: /usr/local/lib/python3.12/dist-packages/opensmile/core/config/egemaps/v01a/eGeMAPSv01a.conf

///////////////////////////////////////////////////////////////////////////////////////
///////// > openSMILE configuration file, Geneva feature set <       //////////////////
/////////                                                            //////////////////
///////// (c) 2014 by audEERING                                      //////////////////
/////////     All rights reserved. See file COPYING for details.     //////////////////
///////////////////////////////////////////////////////////////////////////////////////

;;;;;;; component list ;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;

[componentInstances:cComponentManager]
instance[dataMemory].type=cDataMemory
printLevelStats=0

;;;;;;;;;;;;;;;;;;;;;;;;;;;; main section ;;;;;;;;;;;;;;;;;;;;;;;;;;;

\{\cm[source{?}:include external source]}
\{../../gemaps/v01a/GeMAPSv01a_core.lld.conf.inc}
\{eGeMAPSv01a_core.lld.conf.inc}
\{../../gemaps/v

Step2

In [20]:
# Keep writing the .conf file for submission
# But note it's for documentation purposes
# The actual extraction uses ComParE_2016

print("""
Note on my_features.conf:
--------------------------
The openSMILE Python wrapper (audeering)
does not support loading custom .conf files
directly via opensmile.Smile().

Our custom config file (my_features.conf)
documents the intended feature pipeline
and can be used with the standalone
openSMILE binary (SMILExtract command).

For the Python extraction pipeline,
we use ComParE_2016 which similarly
differs from eGeMAPSv02 by including:
  - More MFCC coefficients
  - Zero Crossing Rate
  - Spectral Flux
  - Chroma features
  - 6373 total features vs 88 in eGeMAPS
""")


Note on my_features.conf:
--------------------------
The openSMILE Python wrapper (audeering)
does not support loading custom .conf files
directly via opensmile.Smile().

Our custom config file (my_features.conf)
documents the intended feature pipeline
and can be used with the standalone
openSMILE binary (SMILExtract command).

For the Python extraction pipeline,
we use ComParE_2016 which similarly
differs from eGeMAPSv02 by including:
  - More MFCC coefficients
  - Zero Crossing Rate
  - Spectral Flux
  - Chroma features
  - 6373 total features vs 88 in eGeMAPS



Step3

In [21]:
smile_custom = opensmile.Smile(
    feature_set   = opensmile.FeatureSet.ComParE_2016,
    feature_level = opensmile.FeatureLevel.Functionals,
)

sample_file = speech_files[0]
df_custom   = smile_custom.process_file(sample_file)

print(f"Feature set   : ComParE_2016")
print(f"Shape         : {df_custom.shape}")
print(f"Feature count : {df_custom.shape[1]}")

Feature set   : ComParE_2016
Shape         : (1, 6373)
Feature count : 6373


In [22]:
all_features = []
all_labels   = []

for path in tqdm(speech_files, desc='Extracting ComParE features'):
    try:
        feats = smile_custom.process_file(path)
        label = get_emotion(path)
        all_features.append(feats.values[0])
        all_labels.append(label)
    except Exception as e:
        print(f"  FAILED: {os.path.basename(path)} — {e}")

print(f"\nExtracted: {len(all_features)} files")

Extracting ComParE features: 100%|██████████| 1440/1440 [07:00<00:00,  3.42it/s]


Extracted: 1440 files


In [26]:
feature_names = df_custom.columns.tolist()

df_bonus = pd.DataFrame(all_features, columns=feature_names)
df_bonus['label'] = all_labels

bonus_csv_path = '/content/drive/MyDrive/AudialMind/ravdess_compare_features.csv'
df_bonus.to_csv(bonus_csv_path, index=False)

print(f"Saved to       : {bonus_csv_path}")
print(f"DataFrame shape: {df_bonus.shape}")
print(f"\nLabel distribution:")
print(df_bonus['label'].value_counts())

Saved to       : /content/drive/MyDrive/AudialMind/ravdess_compare_features.csv
DataFrame shape: (1440, 6374)

Label distribution:
label
calm         192
happy        192
sad          192
angry        192
disgust      192
fearful      192
surprised    192
neutral       96
Name: count, dtype: int64


In [27]:
X = df_bonus.drop(columns=['label']).values
y = df_bonus['label'].values

# Standardise
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 10-fold Stratified CV
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

acc_scores = []
uar_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y), 1):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx],        y[val_idx]

    svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    uar = recall_score(y_val, y_pred,
                       average='macro', zero_division=0)

    acc_scores.append(acc)
    uar_scores.append(uar)
    print(f"  Fold {fold:>2} — Acc: {acc:.4f}  UAR: {uar:.4f}")

print(f"\nComParE Features SVM Results:")
print(f"  Mean Accuracy : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
print(f"  Mean UAR      : {np.mean(uar_scores):.4f} ± {np.std(uar_scores):.4f}")

  Fold  1 — Acc: 0.6736  UAR: 0.6760
  Fold  2 — Acc: 0.6736  UAR: 0.6704
  Fold  3 — Acc: 0.6667  UAR: 0.6796
  Fold  4 — Acc: 0.7431  UAR: 0.7405
  Fold  5 — Acc: 0.7153  UAR: 0.6908
  Fold  6 — Acc: 0.7708  UAR: 0.7602
  Fold  7 — Acc: 0.6944  UAR: 0.7058
  Fold  8 — Acc: 0.6597  UAR: 0.6281
  Fold  9 — Acc: 0.7014  UAR: 0.6918
  Fold 10 — Acc: 0.7708  UAR: 0.7566

ComParE Features SVM Results:
  Mean Accuracy : 0.7069 ± 0.0397
  Mean UAR      : 0.7000 ± 0.0396


In [29]:

egemaps_uar  = 0.6632
compare_uar  = np.mean(uar_scores)
compare_acc  = np.mean(acc_scores)

print("=" * 55)
print("Feature Set Comparison")
print("=" * 55)
print(f"  eGeMAPS   (88 features)    UAR : {egemaps_uar:.4f}")
print(f"  ComParE   (6373 features)  UAR : {compare_uar:.4f}")
print(f"  ComParE   Accuracy         Acc : {compare_acc:.4f}")
print("=" * 55)

if compare_uar > egemaps_uar:
    print("  ComParE BEATS eGeMAPS ✅")
else:
    diff = egemaps_uar - compare_uar
    print(f"  eGeMAPS still better by {diff:.4f}")
    print("  More features ≠ better performance")
    print("  eGeMAPS is research-optimised for emotion")

Feature Set Comparison
  eGeMAPS   (88 features)    UAR : 0.6632
  ComParE   (6373 features)  UAR : 0.7000
  ComParE   Accuracy         Acc : 0.7069
  ComParE BEATS eGeMAPS ✅
